# Akkadian-to-English: Fine-tune ByT5 on 2×T4

**Why previous attempts failed:**
- HF Trainer with 2 GPUs defaults to `DataParallel`, which **replicates** the full model on each GPU
- 581M params × fp32 = 2.2 GB model + 4.4 GB AdamW states + 2.2 GB grads = ~9 GB just for model overhead
- That leaves only ~5.5 GB for activations — not enough for ByT5 at seq=256

**Solution:** Force single-GPU training with `CUDA_VISIBLE_DEVICES=0`.  
Use **Adafactor** (no momentum states = saves 4.4 GB) + **gradient checkpointing** + **fp16** autocast.  
Memory: 2.2 GB model + ~0 adafactor + 2.2 GB grads + grad_ckpt activations ≈ 6-7 GB → fits T4's 14.5 GB.

In [ ]:
import os
# CRITICAL: Use only 1 GPU to avoid DataParallel replication OOM
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import gc, re, warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from datasets import Dataset
import evaluate

warnings.filterwarnings('ignore')
gc.collect(); torch.cuda.empty_cache()

print(f"Visible GPUs: {torch.cuda.device_count()}")
print(f"GPU 0: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

2026-02-03 04:39:32.180230: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770093572.201820    1018 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770093572.208486    1018 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770093572.225247    1018 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770093572.225266    1018 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770093572.225269    1018 computation_placer.cc:177] computation placer alr

Visible GPUs: 1
GPU 0: Tesla T4


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [ ]:
# === CONFIG ===
MODEL_PATH = "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x"
TRAIN_PATH = "/kaggle/input/new-additional-data/train_final.csv"
VAL_PATH   = "/kaggle/input/new-additional-data/val_final.csv"
OUTPUT_DIR  = "./akkadian-finetuned"

MAX_SRC = 256
MAX_TGT = 256
BATCH   = 2           # per device
ACCUM   = 32          # effective = 2 * 32 = 64
EPOCHS  = 3
LR      = 2e-5
PREFIX  = "translate Akkadian to English: "

Path(OUTPUT_DIR).mkdir(exist_ok=True)

In [ ]:
# Load data
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
print(f"Train: {len(train_df):,}  Val: {len(val_df):,}")

In [ ]:
# Preprocessing
def clean_akk(text):
    if pd.isna(text): return ""
    text = str(text)
    text = re.sub(r'\[\.+\]|\[x+\]|\.{3,}|…+', '<gap>', text)
    text = re.sub(r'[!?]', '', text)
    text = re.sub(r'\[([^\]]+)\]', r'\1', text)
    text = re.sub(r'[˹⌈]([^˺⌉]+)[˺⌉]', r'\1', text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_eng(text):
    if pd.isna(text): return ""
    return re.sub(r'\s+', ' ', re.sub(r'"+', '"', str(text))).strip()

In [ ]:
# Load model in fp32 — the Trainer's fp16 flag uses autocast which keeps
# master weights in fp32 and does forward/backward in fp16.
# This is the CORRECT way. Loading in fp16 directly breaks gradient unscaling.
gc.collect(); torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, torch_dtype=torch.float32)
model.gradient_checkpointing_enable()  # trade compute for memory

print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Gradient checkpointing: ON")

In [ ]:
# Tokenize
def make_dataset(df):
    srcs = [PREFIX + clean_akk(t) for t in df['transliteration']]
    tgts = [clean_eng(t) for t in df['translation']]
    enc = tokenizer(srcs, max_length=MAX_SRC, truncation=True, padding=False)
    with tokenizer.as_target_tokenizer():
        lab = tokenizer(tgts, max_length=MAX_TGT, truncation=True, padding=False)
    enc['labels'] = lab['input_ids']
    return Dataset.from_dict(enc)

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)
print(f"Tokenized — Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
# Training setup
# Key choices:
# - adafactor: no momentum states → saves ~4.4 GB vs AdamW
# - gradient_checkpointing: recompute activations → saves ~40% activation memory
# - fp16: forward/backward in fp16, master weights in fp32 → 2x faster, same quality
# - batch=2, accum=32: effective batch=64, low memory per step

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, pad_to_multiple_of=8)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    optim="adafactor",
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=25,
    report_to="none",
    predict_with_generate=False,
    dataloader_num_workers=0,
    remove_unused_columns=True,
    max_grad_norm=1.0,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Trainer ready | 1 GPU | Effective batch: {BATCH * ACCUM}")
print(f"Optimizer: Adafactor | FP16: ON | Grad Checkpointing: ON")
if torch.cuda.is_available():
    print(f"GPU memory before train: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Train
print("="*50)
print("TRAINING")
print("="*50)

result = trainer.train()

print("="*50)
print(f"DONE | Loss: {result.metrics['train_loss']:.4f} | Epochs: {result.metrics.get('epoch', 0):.1f}")
print("="*50)

In [ ]:
# Save
save_path = f"{OUTPUT_DIR}/final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Saved to {save_path}")

In [ ]:
# Evaluate with BLEU / chrF++
print("Evaluating...")
gc.collect(); torch.cuda.empty_cache()

bleu_m = evaluate.load("sacrebleu")
chrf_m = evaluate.load("chrf")

model.eval()
device = next(model.parameters()).device

all_preds, all_refs = [], []
for i in range(0, len(val_df), 8):
    batch = val_df.iloc[i:i+8]
    srcs = [PREFIX + clean_akk(t) for t in batch['transliteration']]
    tgts = [clean_eng(t) for t in batch['translation']]
    
    enc = tokenizer(srcs, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC)
    enc = {k: v.to(device) for k, v in enc.items()}
    
    with torch.no_grad(), torch.amp.autocast('cuda'):
        out = model.generate(**enc, max_new_tokens=MAX_TGT, num_beams=4, length_penalty=1.5)
    
    all_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    all_refs.extend([[t] for t in tgts])
    if i % 200 == 0: print(f"  {i}/{len(val_df)}")

b = bleu_m.compute(predictions=all_preds, references=all_refs)['score']
c = chrf_m.compute(predictions=all_preds, references=all_refs, word_order=2)['score']
g = np.sqrt(b * c) if b > 0 and c > 0 else 0

print(f"\nBLEU: {b:.2f} | chrF++: {c:.2f} | GeomMean: {g:.2f}")

In [ ]:
# Samples
for i in range(min(5, len(all_preds))):
    print(f"\n--- Sample {i+1} ---")
    print(f"Pred: {all_preds[i][:120]}")
    print(f"Ref:  {all_refs[i][0][:120]}")

In [ ]:
gc.collect(); torch.cuda.empty_cache()
print("Done!")